# 简单问答 Agent（QA）：输入问题，输出简洁答案

## 概述
这一节实现一个最小的 Question-Answering（QA）Agent：输入一个问题，输出一个**清晰、简洁**的答案。

## 动机
一个最小 QA Agent 是很多系统的基础模块：你可以先把“如何让模型稳定回答问题”跑通，再逐步叠加检索、工具调用、评估等更复杂的能力。

## 关键组件
- **Language Model**：负责理解问题并生成回答
- **Prompt Template**：约束回答风格与边界（简洁、不要胡编）
- **LangGraph**：把调用模型这件事包装成一个可复用的图（graph）节点

## 方法细节
1) 定义 prompt 模板（system + human）
2) 定义最小 LangGraph：读取最后一个用户问题 → 调用模型 → 把 `AIMessage` 追加回 state 的 `messages`
3) 调用 `app.invoke(...)` 得到回答，并观察 `AIMessage` 的完整元信息（token 用量、finish_reason、tool_calls 等）


### 导入必要的库

In [1]:
from __future__ import annotations

import json
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


load_dotenv("../.env")

True

### 初始化语言模型

In [2]:
llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0,
)

### 定义提示词模板

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant. Answer the user's question clearly and concisely. "
            "If you are not sure, say you don't know.",
        ),
        ("human", "User's question: {question}\n\nPlease provide a clear and concise answer."),
    ]
)

### 构建最小 LangGraph QA

In [4]:
class State(TypedDict):
    # 用 reducer 追加消息（而不是覆盖）
    messages: Annotated[list, add_messages]


def answer_question(state: State) -> dict:
    # 约定：最后一条消息就是用户问题
    last = state["messages"][-1]
    question = getattr(last, "content", "")

    # 用 prompt 生成真正喂给模型的 messages（不把 system message 存进 state）
    messages_to_model = prompt.format_messages(question=question)

    response = llm.invoke(messages_to_model)
    return {"messages": [response]}


builder = StateGraph(State)
builder.add_node("answer", answer_question)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)

app = builder.compile()

### 定义 `get_answer` 函数

In [5]:
def get_answer(question: str) -> str:
    """Get an answer to the given question using the QA app."""

    result = app.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

### 示例

In [6]:
question = "What is the capital of France?"
answer = get_answer(question)
print(f"Question: {question}")
print(f"Answer: {answer}")

Question: What is the capital of France?
Answer: The capital of France is **Paris**.


### 查看 AIMessage 的完整信息（metadata + tool_calls）

除了 `content` 以外，`AIMessage` 里通常还包含 token 用量、finish_reason、请求 id 等元信息。这里把最后一条 `AIMessage` 完整 dump 出来看看。

In [7]:
result = app.invoke({"messages": [{"role": "user", "content": "Hi! My name is Sally."}]})
last = result["messages"][-1]

dump = last.model_dump() if hasattr(last, "model_dump") else {"content": getattr(last, "content", "")}
print(json.dumps(dump, ensure_ascii=False, indent=2))

{
  "content": "Hi Sally! It's nice to meet you. How can I help you today?",
  "additional_kwargs": {
    "refusal": null
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 89,
      "prompt_tokens": 53,
      "total_tokens": 142,
      "completion_tokens_details": {
        "accepted_prediction_tokens": null,
        "audio_tokens": null,
        "reasoning_tokens": 70,
        "rejected_prediction_tokens": null
      },
      "prompt_tokens_details": {
        "audio_tokens": null,
        "cached_tokens": 0
      }
    },
    "model_provider": "openai",
    "model_name": "deepseek-v4-flash-0731",
    "system_fingerprint": null,
    "id": "chatcmpl-cb107c4f-81c2-924f-84ad-b9ef7a725b3e",
    "finish_reason": "stop",
    "logprobs": null
  },
  "type": "ai",
  "name": null,
  "id": "lc_run--019febcc-66e5-7ef2-a275-43d2d6b0f4b4-0",
  "tool_calls": [],
  "invalid_tool_calls": [],
  "usage_metadata": {
    "input_tokens": 53,
    "output_tokens": 89,
    "total_

### 交互式提问

In [8]:
user_question = input("Enter your question: ")
user_answer = get_answer(user_question)
print(f"Answer: {user_answer}")

Answer: 你好！我很好，谢谢你的关心。作为一个AI助手，我没有情感，但我会尽力为你提供帮助。你今天过得怎么样？有什么我可以帮你的吗？
